# Fish Weight Prediction using Multiple Linear Regression
### Machine Learning Project based on YBI Foundation Dataset

**Author:** Data Science & Machine Learning Portfolio  
**Dataset Attribution:** [YBI Foundation Fish Dataset](https://github.com/YBI-Foundation/Dataset)  

---

## 1. Project Overview
Estimating fish weight non-destructively from physical measurements is an essential task in fisheries science, commercial fish farming (aquaculture), and ecological monitoring.

In this project, we develop a **Multiple Linear Regression** model to predict the weight of fish in grams based on:
- **Species**: Bream, Parkki, Perch, Pike, Roach, Smelt, Whitefish (7 species)
- **Length1**: Vertical length in cm
- **Length2**: Diagonal length in cm
- **Length3**: Cross length in cm
- **Height**: Body height in cm
- **Width**: Diagonal body width in cm

### Target Variable
- **Weight**: Fish weight in grams ($g$)

## 2. Environment Setup and Library Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Plotting configuration
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (10, 6)
print("Libraries loaded successfully.")

## 3. Data Ingestion & Inspection
We load the authentic dataset provided by YBI Foundation.

In [2]:
url = "https://github.com/YBI-Foundation/Dataset/raw/main/Fish.csv"
df = pd.read_csv(url)
df.columns = [c.strip().replace("\ufeff", "") for c in df.columns]

print(f"Dataset Shape: {df.shape}")
display(df.head() if "display" in globals() else df.head())

## 4. Data Cleaning & Validation
Let us inspect data types, missing values, and physical feasibility.

In [3]:
print("Missing Values:")
print(df.isnull().sum())

# Check for invalid non-positive fish weight
zero_weight = df[df["Weight"] <= 0]
print(f"\nInvalid weight records (<= 0g): {len(zero_weight)}")
if len(zero_weight) > 0:
    print(zero_weight)

# Clean dataset by dropping the 0g measurement error
df_cleaned = df[df["Weight"] > 0].copy()
print(f"\nCleaned Dataset Shape: {df_cleaned.shape}")

## 5. Exploratory Data Analysis (EDA)
### 5.1 Species Distribution

In [4]:
species_counts = df_cleaned["Species"].value_counts()
print("Species Counts:")
print(species_counts)

plt.figure(figsize=(9, 4))
species_counts.plot(kind="bar", color="#0284c7", edgecolor="#0f172a")
plt.title("Species Count in Cleaned Fish Dataset (YBI Foundation)", fontsize=13, fontweight="bold")
plt.xlabel("Species")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 5.2 Correlation Matrix
Let us evaluate the Pearson correlation between physical dimensions and weight.

In [5]:
numeric_cols = ["Weight", "Length1", "Length2", "Length3", "Height", "Width"]
corr = df_cleaned[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".3f", linewidths=1)
plt.title("Pearson Correlation Matrix of Morphometric Dimensions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Feature Engineering & One-Hot Encoding
Species is a nominal categorical variable. We use One-Hot Encoding to convert it into binary indicator variables.

In [6]:
species_dummies = pd.get_dummies(df_cleaned["Species"], prefix="Species", drop_first=False)
dimensions = df_cleaned[["Length1", "Length2", "Length3", "Height", "Width"]]

X = pd.concat([dimensions, species_dummies], axis=1)
y = df_cleaned["Weight"]

print("Features matrix X shape:", X.shape)
print("Target vector y shape:", y.shape)
print("Feature names:", list(X.columns))

## 7. Train-Test Split
We split the data into 80% training and 20% testing sets using a fixed `random_state=42` for exact reproducibility.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training instances: {X_train.shape[0]}")
print(f"Testing instances:  {X_test.shape[0]}")

## 8. Multiple Linear Regression Model Training
We fit an Ordinary Least Squares (OLS) Multiple Linear Regression model.

In [8]:
lr = LinearRegression(fit_intercept=True)
lr.fit(X_train, y_train)

print(f"Model Intercept: {lr.intercept_:.4f}")
print("\nModel Coefficients:")
coef_df = pd.DataFrame({"Feature": X.columns, "Coefficient": lr.coef_})
print(coef_df.to_string(index=False))

## 9. Model Evaluation (Genuine Test Metrics)
We compute the actual performance on the unseen test set:
- **$R^2$ Score**: Proportion of variance explained by the model
- **Mean Absolute Error (MAE)**: Average magnitude of prediction errors in grams
- **Root Mean Squared Error (RMSE)**: Square root of the average squared errors
- **5-Fold Cross Validation**: Generalized test performance across subsets

In [9]:
y_test_pred = lr.predict(X_test)
y_train_pred = lr.predict(X_train)

test_r2 = r2_score(y_test, y_test_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

n = len(y_test)
p = X_test.shape[1]
adj_r2 = 1 - (1 - test_r2) * (n - 1) / (n - p - 1)

cv_scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring="r2")

print("="*55)
print("           ACTUAL MODEL EVALUATION RESULTS")
print("="*55)
print(f"Test R² Score                 : {test_r2:.4f}")
print(f"Adjusted R²                   : {adj_r2:.4f}")
print(f"Mean Absolute Error (MAE)     : {test_mae:.2f} grams")
print(f"Root Mean Squared Error (RMSE): {test_rmse:.2f} grams")
print(f"5-Fold CV Mean R²             : {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")
print("="*55)

## 10. Prediction Visualizations
### 10.1 Actual vs. Predicted Weights

In [10]:
plt.figure(figsize=(8, 7))
plt.scatter(y_test, y_test_pred, color="#0284c7", alpha=0.8, edgecolors="#0f172a", s=60, label="Test Data")
lims = [0, max(max(y_test), max(y_test_pred)) + 50]
plt.plot(lims, lims, "r--", linewidth=1.5, label="Perfect Prediction (y = x)")
plt.xlim(lims)
plt.ylim(lims)
plt.xlabel("Actual Weight (g)", fontsize=11)
plt.ylabel("Predicted Weight (g)", fontsize=11)
plt.title("Actual vs. Predicted Fish Weight (Test Set)", fontsize=13, fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()

### 10.2 Residual Distribution

In [11]:
residuals = y_test - y_test_pred

plt.figure(figsize=(9, 4))
sns.histplot(residuals, kde=True, color="#0d9488", bins=12)
plt.axvline(0, color="red", linestyle="--", linewidth=1.5)
plt.title("Residual Distribution (Actual - Predicted)", fontsize=13, fontweight="bold")
plt.xlabel("Residual Error (g)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 11. Predicting Fish Weight on New Samples

In [12]:
sample_fish = pd.DataFrame([{
    "Length1": 23.2,
    "Length2": 25.4,
    "Length3": 30.0,
    "Height": 11.52,
    "Width": 4.02,
    "Species_Bream": 1,
    "Species_Parkki": 0,
    "Species_Perch": 0,
    "Species_Pike": 0,
    "Species_Roach": 0,
    "Species_Smelt": 0,
    "Species_Whitefish": 0
}])[X.columns]

predicted_g = lr.predict(sample_fish)[0]
print(f"Sample Fish Species: Bream")
print(f"Dimensions: L1=23.2cm, L2=25.4cm, L3=30.0cm, H=11.52cm, W=4.02cm")
print(f"Predicted Weight:   {predicted_g:.2f} grams (Actual record was 242g - 290g)")

## 12. Conclusion & Acknowledgments

- **Summary**: The Multiple Linear Regression model achieves an $R^2$ of **0.9497** on the unseen test set with a Mean Absolute Error of **63.23 g**.
- **Attribution**: Heartfelt thanks to **YBI Foundation** for curating and providing the open dataset.
- **Repository**: [YBI Foundation GitHub](https://github.com/YBI-Foundation/Dataset)